<a href="https://colab.research.google.com/github/M-Prajana/BA/blob/main/Auto_CoT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip uninstall -y transformers accelerate peft safetensors bitsandbytes
%pip install torch==2.2.0
%pip install transformers==4.35.2
%pip install accelerate==0.24.1
%pip install peft==0.7.1
%pip install bitsandbytes==0.41.3
%pip install safetensors==0.4.0


Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
Found existing installation: safetensors 0.6.2
Uninstalling safetensors-0.6.2:
  Successfully uninstalled safetensors-0.6.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 780.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ==================== GPU-SAFE AUTO-COT GENERATOR WITH SAMPLING ====================
# Generates Chain-of-Thought (Auto-CoT) reasoning from a random sample
# of the last 2 lakh rows, processing in reverse order with GPU optimization.
# ================================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import time, gc

# ---------------- CONFIG ----------------
MODEL_NAME = "EleutherAI/gpt-neo-1.3B"
INPUT_FILE = "/content/all_prompts_CoT_structured.xlsx"  # Your Excel file
OUTPUT_FILE = "/content/prompts_sampled_results.csv"    # Save as CSV for GPU safety
SAMPLE_SIZE = 1000          # How many prompts to randomly pick
BATCH_SIZE = 10             # Adjust based on GPU memory
MAX_NEW_TOKENS = 150        # Max reasoning length
SAVE_INTERVAL = 10          # Save every 10 batches
# ----------------------------------------

# Load model + tokenizer (GPU ready)
print("Loading model on GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float32  # safer for stability than float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype)
model.to(device)
model.eval()
print(f"Model loaded successfully on {device.upper()}!")

# Load data
print("Loading input file...")
df = pd.read_excel(INPUT_FILE)

# Take last 2 lakh rows
df_last = df.tail(200000)

# Randomly sample SAMPLE_SIZE prompts
sample_df = df_last.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Add column for Auto-CoT if missing
if "auto_cot" not in sample_df.columns:
    sample_df["auto_cot"] = ""

# Function to generate Auto-CoT reasoning
def generate_auto_cot(prompt, max_new_tokens=MAX_NEW_TOKENS):
    input_text = f"Q: {prompt}\nLet's think step by step."
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # enable sampling
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Process in reverse order
start_time = time.time()
num_rows = len(sample_df)
batches = range(0, num_rows, BATCH_SIZE)

print(f"Starting Auto-CoT generation for {num_rows} sampled prompts...")

for batch_idx, start in enumerate(batches):
    end = min(start + BATCH_SIZE, num_rows)
    print(f"\nGenerating Auto-CoT for sampled rows {start}–{end - 1} (reverse order)...")

    # Process in reverse within batch
    for i in reversed(range(start, end)):
        prompt = sample_df.loc[i, "Question"]
        if not isinstance(prompt, str) or prompt.strip() == "":
            sample_df.loc[i, "auto_cot"] = ""
            continue
        try:
            reasoning = generate_auto_cot(prompt)
            sample_df.loc[i, "auto_cot"] = reasoning
        except Exception as e:
            print(f"Error at row {i}: {e}")
            sample_df.loc[i, "auto_cot"] = f"Error: {e}"

    # Free memory
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # Save progress periodically
    if (batch_idx + 1) % SAVE_INTERVAL == 0 or end == num_rows:
        sample_df.to_csv(OUTPUT_FILE, index=False)
        print(f"Progress saved up to sampled row {end - 1}")

print("\nAuto-CoT generation completed!")
print(f"Total time: {(time.time() - start_time)/60:.2f} minutes")
print(f"Final file saved as: {OUTPUT_FILE}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

Loading model on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node in

Model loaded successfully on CPU!
Loading input file...
Starting Auto-CoT generation for 1000 sampled prompts...

Generating Auto-CoT for sampled rows 0–9 (reverse order)...

Generating Auto-CoT for sampled rows 10–19 (reverse order)...

Generating Auto-CoT for sampled rows 20–29 (reverse order)...

Generating Auto-CoT for sampled rows 30–39 (reverse order)...

Generating Auto-CoT for sampled rows 40–49 (reverse order)...

Generating Auto-CoT for sampled rows 50–59 (reverse order)...

Generating Auto-CoT for sampled rows 60–69 (reverse order)...

Generating Auto-CoT for sampled rows 70–79 (reverse order)...

Generating Auto-CoT for sampled rows 80–89 (reverse order)...

Generating Auto-CoT for sampled rows 90–99 (reverse order)...
Progress saved up to sampled row 99

Generating Auto-CoT for sampled rows 100–109 (reverse order)...

Generating Auto-CoT for sampled rows 110–119 (reverse order)...

Generating Auto-CoT for sampled rows 120–129 (reverse order)...

Generating Auto-CoT for sam